# Musanga V4 — NB00: New 65-plot field reference, UAV crown validation and modality inventory

**2026-11 V4 restart:** Sections 00–05 retain the working RGB/DSM/MS/LAS metadata audits from V3_1. Sections **06–08** introduce the authoritative new exhaustive field plots and Nadine's class-labelled plot crowns. Run those sections before NB01. No U-Net training occurs in NB00.

**Scientific distinction:** `pct_BA_musanga` is DBH-derived *relative stem basal area*, an independent structural-dominance proxy—not measured crown projected area. Compare to Musanga crown area / whole field plot and, separately, Musanga crown area / all digitized tree crowns.


## 00 — Environment and read-only source registries

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas numpy matplotlib laspy
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json, hashlib, re, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from IPython.display import display
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT_CANDIDATES=[
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final'),
]
ROOT=next((p for p in ROOT_CANDIDATES if p.exists()),None)
assert ROOT is not None, f'Could not find Final in {ROOT_CANDIDATES}'
DATA=ROOT/'Data'
assert DATA.exists(),f'Data is missing: {DATA}'
V31=ROOT/'Outputs/V3_1'
V31_MANIFEST=V31/'00_input_audit/00_input_manifest.json'
assert V31_MANIFEST.exists(),f'Approved V3_1 input manifest missing: {V31_MANIFEST}'
base=json.loads(V31_MANIFEST.read_text())
assert base.get('schema_version')=='v3_1_input_audit_kvo_final', base.get('schema_version')
OUT=ROOT/'Outputs/v4/00_input_audit'
OUT.mkdir(parents=True,exist_ok=True)
print('DATA (read-only):',DATA)
print('V3_1 manifest (read-only):',V31_MANIFEST)
print('V4 output:',OUT)
print('V3_1 annotation reference:',base.get('reference_version'))


## 01 — Discover RGB mosaics, DSM rasters and LAS without guessing site mappings

In [ ]:
FORESTS=('Baego','MOS','Yangambi','Yoko')
# Candidates only. Nothing is approved from filename matching alone.
SITE_KEYWORDS={
    'Baego': ('baego',),
    'MOS': ('mos_', 'mos-', 'mos1', 'mos5', 'mos 1', 'mos 5', 'mlt_mos'),
    'Yangambi': ('yangambi','carlier2020','ybjoseph','ybforgeo'),
    'Yoko': ('yoko',),
}
RGB_NAMES=[
    'Baego.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif',
    'Yangambi_georef.tif',
    'Yoko_kvo_georef.tif',
]
LAS_EXPECTED={
    'MOS': ['mlt_Mos_1_2_Green_densified_point_cloud_georef.las',
            'mlt_Mos_3_4_Green_densified_point_cloud_georef.las',
            'mlt_Mos_5_6_Green_densified_point_cloud_georef.las'],
    'Yangambi': ['Carlier2020_Aug20_flights_group1_densified_point_cloud_part_'+str(i)+'_georef.las' for i in range(1,7)]
              + ['YBJoseph_Green_densified_point_cloud_georef.las',
                 'YBForgeo_Green_densified_point_cloud_georef.las'],
    'Yoko': ['Yoko_survey_georef.las'],
    'Baego': [
        'Nadine_Baego_B2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_B3mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C3mlt_2023_Green_densified_point_cloud.las',
    ],
}
# Bounded to project Data; metadata-only scan, no heavy raster/LAS loads.
paths=[p for p in DATA.rglob('*') if p.is_file() and p.suffix.lower() in ('.tif','.tiff','.las','.laz')]
rows=[]
for p in paths:
    name=p.name.lower()
    tags=[site for site, keys in SITE_KEYWORDS.items() if any(k in name or k in str(p.parent).lower() for k in keys)]
    rows.append({'path':str(p),'name':p.name,'type':'LAS' if p.suffix.lower() in ('.las','.laz') else 'raster',
                 'site_candidate':'|'.join(tags) if tags else '',
                 'dsm_candidate':bool(re.search(r'(^|[_\\-])(dsm|surface|elevation)([_\\-]|$)',p.stem,re.I)),
                 'bytes':p.stat().st_size})
inventory=pd.DataFrame(rows,columns=['path','name','type','site_candidate','dsm_candidate','bytes'])
inventory.to_csv(OUT/'00_file_inventory.csv',index=False)
display(inventory.groupby(['type','site_candidate'],dropna=False).size().rename('n').reset_index())
print('Rasters with DSM-style names:')
display(inventory.query('dsm_candidate').loc[:,['site_candidate','name','path']])
print('Known V3_1 active RGB mosaics:')
for name in RGB_NAMES:
    matches=inventory[inventory.name.str.lower()==name.lower()]
    print(name,'=>',matches.path.tolist())


## 02 — Fast raster metadata audit (no pixel reads)

Read TIFF headers only. Do not downsample massive UAV mosaics over Google Drive in NB00.


In [ ]:
def raster_info(path):
    # FAST: TIFF metadata only. No ds.read(), no resampling, no large Drive I/O.
    with rasterio.open(path) as ds:
        b=ds.bounds
        return {'path':str(path),'name':Path(path).name,
                'crs':str(ds.crs) if ds.crs else None,
                'width':ds.width,'height':ds.height,'bands':ds.count,
                'px_x':ds.transform.a,'px_y':ds.transform.e,
                'left':b.left,'bottom':b.bottom,'right':b.right,'top':b.top,
                'nodata':ds.nodata,'dtype':str(ds.dtypes[0]),
                'block_shapes':str(ds.block_shapes[:2]),
                'sample_valid_fraction':np.nan,
                'sample_z_min':np.nan,'sample_z_p50':np.nan,'sample_z_p99':np.nan}
raster_rows=[]
for p in inventory.query("type == 'raster'").path:
    try:
        raster_rows.append({**raster_info(p),'error':''})
    except Exception as exc:
        raster_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
raster_df=pd.DataFrame(raster_rows)
raster_df.to_csv(OUT/'00_raster_metadata.csv',index=False)
display(raster_df[['name','crs','px_x','px_y','bands','width','height','error']])
print('Metadata audit complete. Valid-data fraction intentionally deferred to small local samples in NB01.')


## 03 — Explicit RGB–DSM file pairing and spatial-overlap audit

Use the five discovered DSMs. This audit checks metadata only: it does not certify within-forest subpixel alignment. In particular, MOS DSMs are coarser than their RGB orthomosaics.


In [ ]:
# Verified filenames from NB00's raster inventory. Locate exact paths in Data; no hardcoded directories.
DSM_BY_RGB={
    'Baego.tif': 'Baego_DSM.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif': 'Mos_1_2_dsm_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif': 'mlt_Mos_5_6_dsm_georef (2).tif',
    'Yangambi_georef.tif': 'Yangambi_georef_DSM.tif',
    'Yoko_kvo_georef.tif': 'Yoko_kvo_georef_DSM.tif',
}
def intersection_fraction(a,b):
    xl=max(a[0],b[0]); yb=max(a[1],b[1]); xr=min(a[2],b[2]); yt=min(a[3],b[3])
    if xr<=xl or yt<=yb:return 0.
    return (xr-xl)*(yt-yb)/max((a[2]-a[0])*(a[3]-a[1]),1e-9)
overlap=[]
for rgb_name,dsm_name in DSM_BY_RGB.items():
    rgb_match=inventory[inventory.name.str.lower()==rgb_name.lower()]
    dsm_match=inventory[inventory.name.str.lower()==dsm_name.lower()]
    rec={'rgb':rgb_name,'rgb_matches':len(rgb_match),'dsm_name':dsm_name,
         'dsm_matches':len(dsm_match),'rgb_path':'','dsm_path':'',
         'crs_match':False,'same_grid':False,'overlap_fraction_of_rgb_bounds':np.nan,
         'extent_qa_pass':False,'registration_verified':False,'reason':''}
    if len(rgb_match)!=1 or len(dsm_match)!=1:
        rec['reason']='Missing/ambiguous RGB or DSM filename; inspect inventory'
    else:
        rec['rgb_path']=rgb_match.iloc[0].path
        rec['dsm_path']=dsm_match.iloc[0].path
        with rasterio.open(rec['rgb_path']) as rgb, rasterio.open(rec['dsm_path']) as dsm:
            if not rgb.crs or not dsm.crs:
                rec['reason']='Missing CRS'
            else:
                rec['crs_match']=bool(rgb.crs==dsm.crs)
                dsm_bounds=transform_bounds(dsm.crs,rgb.crs,*dsm.bounds,densify_pts=21)
                rec['overlap_fraction_of_rgb_bounds']=intersection_fraction(rgb.bounds,dsm_bounds)
                rec['same_grid']=bool(rgb.crs==dsm.crs and rgb.width==dsm.width
                                      and rgb.height==dsm.height
                                      and np.allclose(tuple(rgb.transform),tuple(dsm.transform),atol=1e-8,rtol=0))
                rec['extent_qa_pass']=bool(rec['overlap_fraction_of_rgb_bounds']>=0.95)
                rec['reason']=('Extent OK; check actual footprint and co-registration' if rec['extent_qa_pass']
                               else 'Extent overlap <95%; inspect intended survey coverage')
    overlap.append(rec)
pair_df=pd.DataFrame(overlap)
pair_df.to_csv(OUT/'00_rgb_dsm_pair_audit.csv',index=False)
display(pair_df[['rgb','dsm_name','crs_match','same_grid','overlap_fraction_of_rgb_bounds',
                 'extent_qa_pass','reason']])
print('Pairing metadata audit only; subpixel geometric registration and valid-pixel overlap are NOT yet verified.')
print('MOS DSM grid resolution differs from RGB; NB01 must use windowed resampling aligned to RGB.')
print('No training or mosaics were modified.')


## 04 — SfM LAS header-only audit, including Yangambi's six Carlier parts

In [ ]:
import laspy
las_rows=[]
for p in inventory.query("type == 'LAS'").path:
    try:
        with laspy.open(p) as lf:
            h=lf.header
            las_rows.append({'path':p,'name':Path(p).name,'point_count':int(h.point_count),
                             'crs':str(h.parse_crs()) if h.parse_crs() else '',
                             'xmin':float(h.mins[0]),'ymin':float(h.mins[1]),'zmin':float(h.mins[2]),
                             'xmax':float(h.maxs[0]),'ymax':float(h.maxs[1]),'zmax':float(h.maxs[2]),
                             'point_format':str(h.point_format.id),'version':str(h.version),'error':''})
    except Exception as exc:
        las_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
las_df=pd.DataFrame(las_rows)
las_df.to_csv(OUT/'00_las_header_inventory.csv',index=False)
display(las_df)
expected=[(site,name) for site,names in LAS_EXPECTED.items() for name in names]
missing=[(site,name) for site,name in expected if name.lower() not in set(inventory.name.str.lower())]
print('Expected LAS missing from local Data/LAS:',missing)
print('Interpretation: SfM surface reconstruction, not penetrating lidar; do NOT infer DTM, CHM, or foliage profiles.')
print('Carlier parts 1–6 are one logical Yangambi dataset. Do not physically merge until CRS, coordinates and overlap pass.')


## 04B — Multispectral inventory and footprint metadata (V4 expansion)

**New optional modality, no V3_1 changes.** Inventory the actual `Final/Data/MS` folder only. Baego contains a 4-band raster (band wavelengths/order **not presumed**); MOS 1–2 and 5–6 contain green/red/red-edge/NIR TIFFs; Yangambi MS covers the **YBJoseph and YBForgeo right-hand survey acquisitions only**, **not** the left-hand Yangambi/Carlier survey. There is **no Yoko MS**. Distinguish survey/acquisition groups rather than assigning right-hand Yangambi MS to all Yangambi.

Header-only CRS, affine grid and rectangular intersection with the RGB reference are audited here. Actual valid overlap `ROI.shp ∩ RGB ∩ MS`, band calibration, band ordering, dates and image-level co-registration must be checked on small windows in NB01 before modelling. **Do not treat bounding-box overlap as an approved paired dataset.**

In [ ]:
# Header-only MS audit: cheap and bounded to Data/MS. Do not read any raster pixels here.
MS_DIR=DATA/'MS'
assert MS_DIR.is_dir(),f'Expected multispectral folder missing: {MS_DIR}'
ms_files=sorted(p for p in MS_DIR.rglob('*') if p.is_file() and p.suffix.casefold() in ('.tif','.tiff'))
def ms_group(p):
    n=p.name.casefold()
    if 'baego' in n:return 'Baego_MS'
    if 'mos_1_2' in n:return 'MOS_1_2_MS'
    if 'mos_5_6' in n:return 'MOS_5_6_MS'
    if 'ybjoseph' in n:return 'Yangambi_YBJoseph_right_MS'
    if 'ybforgeo' in n:return 'Yangambi_YBForgeo_right_MS'
    return 'UNASSIGNED'
def ms_band_hint(p):
    n=p.stem.casefold()
    if 'red edge' in n or 'red_edge' in n or 'rededge' in n:return 'red_edge'
    for band in ('green','nir','red'):
        if re.search(r'(^|[_ ])'+band+r'([_ ]|$)',n):return band
    return 'multi_band_unverified' if 'baego' in n else 'unassigned'
ms_meta=[]
for p in ms_files:
    try:
        with rasterio.open(p) as ds:
            b=ds.bounds
            ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                name=p.name,path=str(p),bands=ds.count,
                band_descriptions=str(ds.descriptions),
                crs=str(ds.crs) if ds.crs else '',
                width=ds.width,height=ds.height,
                pixel_x_m=float(ds.transform.a),pixel_y_m=float(ds.transform.e),
                left=b.left,bottom=b.bottom,right=b.right,top=b.top,
                nodata=ds.nodata,dtype=str(ds.dtypes[0]),error=''))
    except Exception as e:
        ms_meta.append(dict(group=ms_group(p),band_hint=ms_band_hint(p),
                            name=p.name,path=str(p),error=repr(e)))
MS_META=pd.DataFrame(ms_meta)
MS_META.to_csv(OUT/'00_ms_raster_headers.csv',index=False)
print('=== MS GROUPS / BANDS FOUND ===')
display(MS_META.groupby(['group','band_hint'],dropna=False).size().rename('files').reset_index())
display(MS_META[['group','band_hint','name','bands','band_descriptions','crs','error']])
expected={'Baego_MS','MOS_1_2_MS','MOS_5_6_MS','Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS'}
missing_groups=expected-set(MS_META.group)
print('Expected groups absent:',sorted(missing_groups))
print('Yoko: NO MS. Yangambi Carlier/left survey: NO MS.')
# Duplicate band types (e.g. MOS1_2 red-edge files) must be resolved, not selected by filename heuristics.
duplicates=MS_META[MS_META.band_hint.ne('multi_band_unverified') &
    MS_META.duplicated(['group','band_hint'],keep=False)]
if len(duplicates):
    print('AMBIGUOUS MS BANDS — inspect files; no automatic choice:')
    display(duplicates[['group','band_hint','name','path']])
# Extent overlap is descriptive only, NOT valid-data support or acquisition identity.
RGB_REFERENCE={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
paired=[]
for group,rgb_name in RGB_REFERENCE.items():
    matches=inventory[inventory.name.str.casefold()==rgb_name.casefold()]
    assert len(matches)==1,('RGB reference ambiguous',rgb_name)
    with rasterio.open(matches.iloc[0].path) as rgb:
        for _,rec in MS_META[MS_META.group==group].iterrows():
            if rec.get('error',''):continue
            if not rec['crs']:
                frac=np.nan
            else:
                bb=transform_bounds(rec['crs'],rgb.crs,rec.left,rec.bottom,rec.right,rec.top,densify_pts=21)
                frac=intersection_fraction(rgb.bounds,bb)
            paired.append(dict(group=group,band_hint=rec.band_hint,name=rec['name'],
                rgb_reference=rgb_name,bbox_fraction_of_rgb=frac,
                rgb_crs=str(rgb.crs),ms_crs=rec.crs,
                same_crs=bool(str(rgb.crs)==rec.crs),
                acquisition_match_verified=False,roi_valid_overlap_verified=False))
MS_OVERLAP=pd.DataFrame(paired)
MS_OVERLAP.to_csv(OUT/'00_ms_rgb_extent_audit.csv',index=False)
display(MS_OVERLAP[['group','band_hint','name','bbox_fraction_of_rgb','same_crs']])
ms_registry=dict(schema='musanga_v4_ms_inventory_1',
    ms_dir=str(MS_DIR),
    groups={group:MS_META.loc[MS_META.group==group,['name','path','band_hint','bands']].to_dict('records')
            for group in sorted(MS_META.group.unique())},
    rgb_reference=RGB_REFERENCE,
    known_absence=['Yoko','Yangambi_Carlier_left'],
    notes=['Yangambi YBJoseph and YBForgeo are right-hand independent acquisitions',
           'Baego 4-band ordering and wavelengths require verification',
           'Band radiometry, date, ROI validity and co-registration not yet verified',
           'MS/DSM/LAS comparisons require matched valid ROI support'])
(OUT/'00_ms_registry.json').write_text(json.dumps(ms_registry,indent=2))
print('Saved MS header audit, extent audit and registry. No large raster reads performed.')


## 05 — Freeze proposed V4 audit contract (no modelling yet)

In [ ]:
audit={
    'schema_version':'v4_nb00_rgb_dsm_sfm_inventory_1',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'inherited_reference_manifest':str(V31_MANIFEST),
    'inherited_reference_schema':base.get('schema_version'),
    'inherited_reference_version':base.get('reference_version'),
    'v3_1_folds_policy':'Reuse frozen 30-circle within-site spatial folds; do not regenerate',
    'v3_1_excluded_original_circle_ids':[6,9,24,25],
    'v3_1_training_exclusion_buffer_m':15,
    'primary_modalities':['RGB','DSM-derived local geometry'],
    'optional_modalities':['MS bands/indices on verified survey overlaps','LAS SfM 3D diagnostics'],
    'ms_inventory_manifest':str(OUT/'00_ms_registry.json'),
    'ms_group_count':int(MS_META.group.nunique()),
    'ms_file_count':int(len(MS_META)),
    'ms_partial_coverage_policy':'Do not extrapolate YBJoseph/YBForgeo MS to Yangambi Carlier left; no Yoko MS',
    'comparison_support_policy':'Matched valid ROI∩RGB∩modality footprint; same frozen folds per comparison',
    'experimental_las':'small stratified crown sample, only if added information beyond DSM',
    'prohibited_unjustified_features':['CHM','DTM','understory profile from SfM'],
    'output_root':str(ROOT/'Outputs/v4'),
    'n_indexed_rasters':int((inventory.type=='raster').sum()),
    'n_indexed_las':int((inventory.type=='LAS').sum()),
    'rgb_dsm_pairings_confirmed':bool(len(pair_df)==len(RGB_NAMES) and (pair_df.rgb_matches.eq(1)&pair_df.dsm_matches.eq(1)&pair_df.crs_match).all()),
    'rgb_dsm_extent_qa_pass':bool(pair_df.extent_qa_pass.all()),
    'rgb_dsm_registration_verified':False,
    'next_step':'N01 after RGB–DSM identity, coverage, registration and dates pass QA',
}
manifest=OUT/'00_v4_input_audit_manifest.json'
manifest.write_text(json.dumps(audit,indent=2))
print('WROTE:',manifest)
print('RGB–DSM filenames/CRS confirmed:',audit['rgb_dsm_pairings_confirmed'])
print('RGB–DSM extent QA:',audit['rgb_dsm_extent_qa_pass'])
print('Subpixel registration still pending:',not audit['rgb_dsm_registration_verified'])
print('V3_1 remains unchanged:',V31_MANIFEST)
if not audit['rgb_dsm_pairings_confirmed']:
    print('ACTION REQUIRED: investigate unmatched filenames/CRS; check 00_rgb_dsm_pair_audit.csv.')


## 06 — V4 authoritative plot and crown datasets (do not reuse old Plot_Data.shp)
The new source files under `Final/Data/Field` are **`Plot_Data_BA_nobuffer_ROI 2.gpkg`** (internal layer `Plot_Data_BA_nobuffer_ROI`) and **`Tree_Annotation_georef_nadine_new 2.shp`**. The plot dataset contains 65 spatially referenced field census plots, and Nadine's annotations are confirmed **exhaustive**, with **Class_ID == 1** = Musanga and all other recorded IDs = non-Musanga. Repair invalid polygons conservatively without changing source files. Plot boundary area must be compared with `surface_ha` before inference; preserve both supplied and reconstructed nominal geometries for sensitivity.

In [ ]:
# NB00.06 — Reference provenance, repair QA and area-support diagnostics
import geopandas as gpd
from shapely import make_valid
from shapely.ops import unary_union
from shapely.geometry import box, GeometryCollection
from scipy.stats import linregress, spearmanr
FIELD_DIR=DATA/'Field'
PLOT_V4=FIELD_DIR/'Plot_Data_BA_nobuffer_ROI 2.gpkg'
CROWN_V4=FIELD_DIR/'Tree_Annotation_georef_nadine_new 2.shp'
for p in [PLOT_V4,CROWN_V4]: assert p.is_file(),f'Missing V4 field source: {p}'
P=gpd.read_file(PLOT_V4,layer='Plot_Data_BA_nobuffer_ROI').to_crs('EPSG:32635')
C=gpd.read_file(CROWN_V4).to_crs('EPSG:32635')
required={'Plot_name','Site','pct_BA_musanga','surface_ha'}
assert required.issubset(P.columns),(required-set(P.columns),P.columns.tolist())
assert 'Class_ID' in C.columns
assert len(P)==65,f'Expected 65 plots, got {len(P)}'
assert P.Plot_name.is_unique
assert P.geometry.notna().all()
assert pd.to_numeric(P.pct_BA_musanga,errors='coerce').between(0,100).all()
C['musanga_label']=pd.to_numeric(C.Class_ID,errors='coerce').eq(1).astype('uint8')
assert pd.to_numeric(C.Class_ID,errors='coerce').notna().all(), 'Unknown class values require explicit review'
print('Field reference:',len(P),'plots,',len(C),'crowns,',int(C.musanga_label.sum()),'Musanga crowns')

def polygon_only(g):
    if g is None or g.is_empty:return None
    if g.geom_type in ('Polygon','MultiPolygon'): return g
    sub=[]
    for v in getattr(g,'geoms',[]):
        if v.geom_type in ('Polygon','MultiPolygon'):sub.append(v)
    return unary_union(sub) if sub else None

repair_rows=[]; geometries=[]
for i,g in enumerate(C.geometry):
    if g is None or g.is_empty:
        geometries.append(None);repair_rows.append((i,'empty',np.nan));continue
    if g.is_valid and g.geom_type in ('Polygon','MultiPolygon'):
        geometries.append(g);repair_rows.append((i,'ok',0.));continue
    try:
        fix=polygon_only(make_valid(g))
        if fix is None or fix.is_empty or not fix.is_valid:
            geometries.append(None);repair_rows.append((i,'quarantine',np.nan));continue
        change=abs(fix.area-g.area)/max(fix.area,g.area,1e-9)
        status='repaired' if change<=0.05 else 'quarantine_gt5pct_area_change'
        geometries.append(fix if status=='repaired' else None)
        repair_rows.append((i,status,change))
    except Exception:
        geometries.append(None);repair_rows.append((i,'quarantine_exception',np.nan))
C['geometry']=geometries
REPAIR_QA=pd.DataFrame(repair_rows,columns=['source_row','status','relative_area_change'])
REPAIR_QA.to_csv(OUT/'00_v4_field_crown_repairs.csv',index=False)
print('Crown topology QA:');display(REPAIR_QA.status.value_counts().rename_axis('status').reset_index(name='n'))
C=C[C.geometry.notna()].copy()
P['mapped_area_ha']=P.geometry.area/10000
P['census_area_ha']=pd.to_numeric(P.surface_ha,errors='coerce')
assert P.census_area_ha.gt(0).all()
P['area_ratio_mapped_to_census']=P.mapped_area_ha/P.census_area_ha
P['area_mismatch']=~np.isclose(P.mapped_area_ha,P.census_area_ha,rtol=.02,atol=.003)
P['field_geometry']=P.geometry
nominal=[]
for g,a in zip(P.geometry,P.census_area_ha):
    half=np.sqrt(a*10000)/2;x,y=g.centroid.x,g.centroid.y
    nominal.append(box(x-half,y-half,x+half,y+half))
P['nominal_geometry']=nominal
# Use mapped shape when verified nominal, otherwise tentative centred nominal square.
P['validation_geom_source']=np.where(P.area_mismatch,'centred_nominal_sensitivity','supplied_plot_roi')
P['geometry']=list(np.where(P.area_mismatch,np.array(nominal,dtype=object),np.array(P.field_geometry.to_list(),dtype=object)))
print('Plot area discrepancies:',int(P.area_mismatch.sum()))
display(P.groupby(['Site','area_mismatch'],dropna=False).size().rename('n').reset_index())
display(P.loc[P.area_mismatch,['Plot_name','Site','mapped_area_ha','census_area_ha','area_ratio_mapped_to_census']].round(3))
P.drop(columns=['field_geometry','nominal_geometry']).to_file(OUT/'00_v4_field_plot_support.gpkg',driver='GPKG')


## 07 — Independent ecological comparison: field basal-area dominance vs exhaustive UAV crown fractions
Compute Musanga projected crown area / full plot area, as well as Musanga projected crown area / digitized total crown area. Clip all crown polygons to the exact same nominal plot support. Union overlapping polygons **within each class** to avoid double counting; audit overlap between class unions and percentage of plot occupied by mapped crowns. The non-Musanga classes are confirmed negative crown objects; canopy gaps are not labelled crowns. The paired field-vs-UAV analysis is a human-reference assessment and does not train or validate a U-Net.

In [ ]:
# NB00.07 — Spatial overlay using only modest plot and crown polygon data
import matplotlib.pyplot as plt
from shapely.strtree import STRtree
from shapely.ops import unary_union
cc=list(C.geometry);cl=C.musanga_label.to_numpy(int)
tree=STRtree(cc)
records=[]
for row in P.itertuples(index=False):
    geom=row.geometry
    idx=tree.query(geom,predicate='intersects')
    pos=[cc[int(i)].intersection(geom) for i in idx if cl[int(i)]==1]
    neg=[cc[int(i)].intersection(geom) for i in idx if cl[int(i)]==0]
    union_pos=unary_union(pos) if pos else GeometryCollection()
    union_neg=unary_union(neg) if neg else GeometryCollection()
    union_both=unary_union([union_pos,union_neg])
    a_pos=float(union_pos.area);a_neg=float(union_neg.area)
    a_all=float(union_both.area);a_plot=float(geom.area)
    records.append(dict(
        Plot_name=row.Plot_name,Site=row.Site,
        field_BA_pct=float(row.pct_BA_musanga),
        musanga_crown_area_m2=a_pos,
        non_musanga_crown_area_m2=a_neg,
        digitized_union_area_m2=a_all,
        overlapping_label_area_m2=float(union_pos.intersection(union_neg).area),
        plot_area_m2=a_plot,
        crown_fraction_plot_pct=100*a_pos/a_plot,
        musanga_fraction_digitized_crowns_pct=100*a_pos/a_all if a_all else np.nan,
        digitized_crown_coverage_pct=100*a_all/a_plot,
        n_musanga_polygons=int(sum(cl[int(i)]==1 for i in idx)),
        n_non_musanga_polygons=int(sum(cl[int(i)]==0 for i in idx)),
        geometry_source=row.validation_geom_source
    ))
FIELD_COMPARE=pd.DataFrame(records)
FIELD_COMPARE.to_csv(OUT/'00_v4_field_BA_vs_Nadine_65plots.csv',index=False)
print('Overlay records:',len(FIELD_COMPARE),'class-overlap plots:',
      int((FIELD_COMPARE.overlapping_label_area_m2>0.1).sum()))
display(FIELD_COMPARE.groupby('Site')[[
    'field_BA_pct','crown_fraction_plot_pct',
    'musanga_fraction_digitized_crowns_pct','digitized_crown_coverage_pct'
]].agg(['size','mean','median']).round(2))

def compare_stats(frame, metric,site):
    z=frame[['field_BA_pct',metric]].replace([np.inf,-np.inf],np.nan).dropna()
    x=z.field_BA_pct.to_numpy(float);y=z[metric].to_numpy(float)
    q=dict(site=site,metric=metric,n=len(z),bias_pp=np.mean(y-x),mae_pp=np.mean(abs(y-x)))
    if len(z)>2 and np.std(x)>0 and np.std(y)>0:
        r=linregress(x,y)
        q.update(Pearson_r=r.rvalue,Pearson_p=r.pvalue,Spearman_rho=spearmanr(x,y).statistic,
                 Regression_R2=r.rvalue**2,slope=r.slope,intercept=r.intercept)
    return q
stats=[]
for site,grp in [('ALL',FIELD_COMPARE),*FIELD_COMPARE.groupby('Site')]:
    for metric in ['crown_fraction_plot_pct','musanga_fraction_digitized_crowns_pct']:
        stats.append(compare_stats(grp,metric,site))
FIELD_STATS=pd.DataFrame(stats)
FIELD_STATS.to_csv(OUT/'00_v4_field_BA_vs_Nadine_statistics.csv',index=False)
display(FIELD_STATS.round(3))
fig,axes=plt.subplots(1,2,figsize=(12.5,5.4),sharex=True,layout='constrained')
for ax,metric,title in zip(axes,[
    'crown_fraction_plot_pct','musanga_fraction_digitized_crowns_pct'
],['Musanga crown / plot area','Musanga crown / all digitized crowns']):
    for site,grp in FIELD_COMPARE.groupby('Site'):
        ax.scatter(grp.field_BA_pct,grp[metric],s=48,alpha=.8,label=f'{site} (n={len(grp)})')
    ax.plot([0,100],[0,100],ls='--',color='gray',lw=1,label='1:1')
    subset=FIELD_COMPARE[['field_BA_pct',metric]].dropna()
    if len(subset)>2 and subset.field_BA_pct.std()>0:
        fit=linregress(subset.field_BA_pct,subset[metric])
        xx=np.array([0,100]);ax.plot(xx,fit.intercept+fit.slope*xx,color='black',lw=1.5)
        ax.text(.03,.97,f'n={len(subset)}, r={fit.rvalue:.3f}, slope={fit.slope:.2f}',
                va='top',transform=ax.transAxes,fontsize=9)
    ax.set(xlim=(0,100),ylim=(0,100),title=title,
           xlabel='Field Musanga basal area (%)',ylabel='Nadine UAV Musanga (%)')
    ax.grid(alpha=.2)
axes[1].legend(fontsize=8,loc='lower right')
fig.suptitle('Exhaustive field-plot annotations: ecological agreement, not segmentation accuracy')
fig.savefig(OUT/'00_v4_field_BA_vs_Nadine_65plots.png',dpi=220,bbox_inches='tight')
fig.savefig(OUT/'00_v4_field_BA_vs_Nadine_65plots.pdf',bbox_inches='tight')
plt.show()
print('Saved 65-plot comparison and plot QA in',OUT)


## 08 — V4 input data contract and controlled modality comparison plan
Ablation sequence: **RGB only** (baseline); RGB + DSM (canopy-surface height, texture and topographic relief); RGB + calibrated MS bands (red, green, red-edge, NIR); RGB + SfM LAS-derived 3D metrics (only if registered, supported and independently informative); and a full multimodal candidate. Keep architecture, pretrained weights, patch footprints, annotation support, balanced site sampling, folds, augmentation and evaluation identical. To isolate modality contribution, compare all models on a **common valid-data subset**; report coverage and site-level missingness separately. LAS is structure-from-motion (not ground-classified LiDAR): do not call its elevation a canopy height model until a DTM/ground reference is validated. Spatially blocked CV must exclude the held-out plots and overlapping KVO circles from training.

In [ ]:
# NB00.08 — Machine-readable reference and modality ablation contract
C31=Path(base['exhaustive_annotations']['positive_polygons'])
R31=Path(base['exhaustive_annotations']['complete_roi_boundaries'])
assert C31.name=='Circles_Annotations_kvo_final.shp'
assert R31.name=='Circles_ROI_NAdine.shp'
contract={
  'schema':'musanga_v4_field65_input_audit_2026_11',
  'plot_file':str(PLOT_V4),'plot_layer':'Plot_Data_BA_nobuffer_ROI',
  'field_fraction_column':'pct_BA_musanga','n_field_plots':int(len(P)),
  'exhaustive_plot_crowns':str(CROWN_V4),
  'crown_musanga_class_id':1,'other_class_ids':'negative_crown_polygons',
  'exhaustive_plot_support':'geometry corrected/flagged in NB00; background not a negative crown object',
  'field_support_qa':str(OUT/'00_v4_field_plot_support.gpkg'),
  'field_BA_comparison':str(OUT/'00_v4_field_BA_vs_Nadine_65plots.csv'),
  'field_BA_stats':str(OUT/'00_v4_field_BA_vs_Nadine_statistics.csv'),
  'circle_positive_crowns':str(C31),'circle_complete_roi':str(R31),
  'partial_site_crowns':base['shared_annotations'],
  'ablation_modalities':['RGB','RGB_DSM','RGB_MS','RGB_LAS_features','RGB_DSM_MS','RGB_DSM_MS_LAS'],
  'modality_policy':'No modality comparison without identical ground support and spatial folds; RGB-only baseline everywhere',
  'splits_policy':'Spatial blocked folds by whole plot; overlapping circles and partial crowns cannot leak into held-out field plots',
  'geometry_exception_policy':'Report original versus census area; use centred nominal reconstruction as sensitivity only',
  'data_root':str(DATA)
}
(OUT/'00_v4_field65_contract.json').write_text(json.dumps(contract,indent=2))
display(pd.DataFrame([{'component':k,'value':str(v)} for k,v in contract.items() if k.endswith('crowns') or k in ['plot_file','n_field_plots','field_fraction_column']]))
print('V4 65-plot field contract ready:',OUT/'00_v4_field65_contract.json')
